In [1]:
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage,SystemMessage
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import DirectoryLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma, FAISS
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langgraph.graph import StateGraph,START, END,MessagesState
from typing import TypedDict,Annotated,Literal
import os 
import sys
sys.path.insert(1, r'D:\Notebooks\LLM\env')
from enviorment import load_env
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_core.tools import tool
#from pydirectoryloader import rag_function
import os 
load_env()
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import GoogleSerperAPIWrapper,WikipediaAPIWrapper
import logging
from langchain.agents.middleware import before_model, after_model
from langchain.agents.middleware import SummarizationMiddleware
#from langchain.middleware.summarization import SummarizationMiddleware
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
#from langchain_anthropic.middleware import anthropicPromptCachingMiddleware
from langchain.agents.middleware import TodoListMiddleware
from langgraph.checkpoint.memory import InMemorySaver




















In [2]:


# Configure the wrapper (optional)
api_wrapper = WikipediaAPIWrapper(
    top_k_results=1,  # Number of Wikipedia pages to fetch
    doc_content_chars_max=1000 # Max characters from each page
)

# Initialize the Wikipedia query tool
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)
checkpoint =InMemorySaver()

In [3]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [4]:
model=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",   # or gemini-2.0-pro, 1.5-pro, etc.
    temperature=0.2
)

C:\Users\Dharmendra Vartika\AppData\Local\Temp\ipykernel_14632\961579943.py:1: DeprecationWarning: max_tokens_before_summary is deprecated. Use trigger=('tokens', value) instead.
  summarization = SummarizationMiddleware(
C:\Users\Dharmendra Vartika\AppData\Local\Temp\ipykernel_14632\961579943.py:1: DeprecationWarning: messages_to_keep is deprecated. Use keep=('messages', value) instead.
  summarization = SummarizationMiddleware(


In [6]:
search_Tavily = TavilySearchResults(max_results=2)

C:\Users\Dharmendra Vartika\AppData\Local\Temp\ipykernel_14632\3156774081.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_Tavily = TavilySearchResults(max_results=2)


In [7]:
import requests
def getweahter(city:str)->dict:
     """"this function is to get weather report for a given city anyhwere in the world"""
     url = f'https://api.weatherstack.com/forecast?access_key=ec07421f5107d04ec9fa783c8fcfd6c5&query={city}'
     response = requests.get(url)
     return response.json()

In [205]:
# from langchain_core.messages import AIMessage
# from langchain_core.runnables import RunnableConfig

# class TodoCompletionMiddleware:
#     def __call__(self, next, *args, **kwargs):
#         async def _call(input, config: RunnableConfig, **kw):
#             result = await next(input, config=config, **kw)

#             todos = config.get("metadata", {}).get("todos", [])
#             if not todos:
#                 return result

#             # Mark all in-progress tasks as completed
#             updated_todos = []
#             for t in todos:
#                 if t["status"] == "in_progress":
#                     updated_todos.append({"content": t["content"], "status": "completed"})
#                 else:
#                     updated_todos.append(t)

#             # Inject updated todos back into metadata
#             config["metadata"]["todos"] = updated_todos

#             return result
#         return _call


In [31]:
summarization = SummarizationMiddleware(
     messages_to_keep=3, 
   #  max_tokens_before_summary=10,
     trigger=("tokens", 200),
     model=model
 )

C:\Users\Dharmendra Vartika\AppData\Local\Temp\ipykernel_14632\1698403663.py:1: DeprecationWarning: messages_to_keep is deprecated. Use keep=('messages', value) instead.
  summarization = SummarizationMiddleware(


In [32]:
agent=create_agent(model=model,tools=[search_Tavily,getweahter,wiki],system_prompt="You are an expert analyst...", middleware=[summarization], checkpointer=checkpoint)

In [26]:
#agent

In [33]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "Can you get me top 20 news of Mumbai and weather of Mumbai"}]},{"configurable": {"thread_id": "1"}})

In [34]:
for i in result['messages']:
    i.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

The user, Bob, requested the top 20 news articles and weather for Mumbai. The assistant has retrieved the current weather for Mumbai (31°C, 41% humidity, 23 km/h wind speed, AQI PM2.5: 76.25, PM10: 76.55, US EPA Index: 4) and a forecast for 2025-12-03 (min 24°C, max 28°C). News articles related to Mumbai have also been retrieved, covering topics such as infrastructure projects, crime, BMC announcements, and aviation delays.
================================== Ai Message ==================================
Tool Calls:
  tavily_search_results_json (9f47789f-bda7-4811-aa41-ce9ad33c6a77)
 Call ID: 9f47789f-bda7-4811-aa41-ce9ad33c6a77
  Args:
    query: top 20 news of Mumbai
  getweahter (864d02b2-4abf-46c0-94c4-d069d4e83a86)
 Call ID: 864d02b2-4abf-46c0-94c4-d069d4e83a86
  Args:
    city: Mumbai
================================= Tool Message ======================

In [35]:
result2 = agent.invoke({
    "messages": [{"role": "user", "content": "Can you get me top 1000 news of Singapore and weather of Singapore"}]},{"configurable": {"thread_id": "1"}})

In [36]:
for i in result2['messages']:
    i.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

The user, Bob, requested the top 20 news articles and weather for Mumbai. The assistant has retrieved the current weather for Mumbai (31°C, 41% humidity, 23 km/h wind speed, AQI PM2.5: 76.25, PM10: 76.55, US EPA Index: 4) and a forecast for 2025-12-03 (min 24°C, max 28°C). News articles related to Mumbai have also been retrieved, covering topics such as infrastructure projects, crime, BMC announcements, and aviation delays. The user has now requested the top 1000 news articles and weather for Singapore.
================================== Ai Message ==================================
Tool Calls:
  tavily_search_results_json (69c962a5-32f2-4e1b-b039-e5e569abdc58)
 Call ID: 69c962a5-32f2-4e1b-b039-e5e569abdc58
  Args:
    query: top 1000 news of Singapore
  getweahter (7ddc456c-493f-4fd8-8c10-c56d32988455)
 Call ID: 7ddc456c-493f-4fd8-8c10-c56d32988455
  Args:


In [37]:
result2 = agent.invoke({
    "messages": [{"role": "user", "content": "what was arushi murder case"}]},{"configurable": {"thread_id": "1"}})

In [39]:
result3 = agent.invoke({
    "messages": [{"role": "user", "content": "what was 1992 scam"}]},{"configurable": {"thread_id": "1"}})

In [38]:
for i in result2['messages']:
    i.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

The user, Bob, initially requested the top 20 news articles and weather for Mumbai. The assistant provided the current weather for Mumbai (31°C, 41% humidity, 23 km/h wind speed, AQI PM2.5: 76.25, PM10: 76.55, US EPA Index: 4) and a forecast for 2025-12-03 (min 24°C, max 28°C). News articles related to Mumbai were also retrieved, covering infrastructure, crime, BMC announcements, and aviation delays. The user has now requested the top 1000 news articles and weather for Singapore.

**Singapore News:**
*   Longer waits of up to 30 minutes for three months during Circle Line tunnel works in 2026
*   Motorists to get S$30 off traffic fines if paid early
*   More than 800 Singaporeans safely return from flood-stricken Hat Yai
*   Singapore secondary schools to forbid smartphone use during recess, CCA hours
*   Over S$7m in fines issued for tax evasion at Singapor

In [40]:
for i in result3['messages']:
    i.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

The user initially requested news and weather for Mumbai, which was provided. The user then requested news and weather for Singapore, which was also provided. The user's last query was about the "Arushi murder case". The assistant used the Wikipedia tool to find information on the "2008 Noida double murder case", which involved the unsolved murders of Aarushi Talwar and Hemraj Banjade. The summary provided details about the case, the initial suspects, and the criticism faced by the police investigation.
================================ Human Message =================================

what was 1992 scam
================================== Ai Message ==================================
Tool Calls:
  wikipedia (bfc36b70-de7e-417d-8907-4ad988e2f6f4)
 Call ID: bfc36b70-de7e-417d-8907-4ad988e2f6f4
  Args:
    query: 1992 Indian stock market scam
====================

Human in the loop middleware

In [41]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

In [43]:

middleware=HumanInTheLoopMiddleware(
    interrupt_on={
        "getweahter": { "allowed_decisions": ["approve", "edit", "reject"]}
    }
)




In [44]:
agent=create_agent(model=model,tools=[search_Tavily,getweahter,wiki],system_prompt="You are an expert analyst...", middleware=[middleware], checkpointer=checkpoint)

In [49]:
result2 = agent.invoke({
    "messages": [{"role": "user", "content": "Can you get me top 10 news of Singapore and weather of Singapore"}]},{"configurable": {"thread_id": "2"}})

In [50]:
for i in result2['messages']:
    i.pretty_print()

================================ Human Message =================================

Can you get me top 10 news of Singapore and weather of Singapore
================================== Ai Message ==================================
Tool Calls:
  tavily_search_results_json (db49e692-b06c-4a4f-b1f1-fb728f88e8c5)
 Call ID: db49e692-b06c-4a4f-b1f1-fb728f88e8c5
  Args:
    query: top 10 news of Singapore
  getweahter (465927f2-8bd0-4fd0-8c6c-50967601d435)
 Call ID: 465927f2-8bd0-4fd0-8c6c-50967601d435
  Args:
    city: Singapore


In [51]:
interrupts = result2["__interrupt__"]
interrupts[0].value

{'action_requests': [{'name': 'getweahter',
   'args': {'city': 'Singapore'},
   'description': "Tool execution requires approval\n\nTool: getweahter\nArgs: {'city': 'Singapore'}"}],
 'review_configs': [{'action_name': 'getweahter',
   'allowed_decisions': ['approve', 'edit', 'reject']}]}

In [52]:
from langgraph.types import Command
response = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}  # or "edit", "reject"
    ), config={"configurable": {"thread_id": "2"}}
)

In [53]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

Can you get me top 10 news of Singapore and weather of Singapore
================================== Ai Message ==================================
Tool Calls:
  tavily_search_results_json (db49e692-b06c-4a4f-b1f1-fb728f88e8c5)
 Call ID: db49e692-b06c-4a4f-b1f1-fb728f88e8c5
  Args:
    query: top 10 news of Singapore
  getweahter (465927f2-8bd0-4fd0-8c6c-50967601d435)
 Call ID: 465927f2-8bd0-4fd0-8c6c-50967601d435
  Args:
    city: Singapore
================================= Tool Message =================================
Name: tavily_search_results_json

[{"title": "Singapore News Today: Breaking Stories & Live Updates - CNA", "url": "https://www.channelnewsasia.com/singapore", "content": "### Longer waits of up to 30 minutes for three months during Circle Line tunnel works in 2026\n\n01/12/2025\n\n###### Motorists to get S$30 off traffic fines if paid early\n\n01/12/2025\n\n###### More than 800 Singaporean

In [55]:
response['messages'][-1]

AIMessage(content="Here are the top 10 news of Singapore and the weather report for Singapore:\n\n**Top 10 News of Singapore:**\n\n*   Longer waits of up to 30 minutes for three months during Circle Line tunnel works in 2026\n*   Motorists to get S$30 off traffic fines if paid early\n*   More than 800 Singaporeans safely return from flood-stricken Hat Yai\n*   Singapore secondary schools to forbid smartphone use during recess, CCA hours\n*   Over S$7m in fines issued for tax evasion at Singapore checkpoints in first 10 months of 2025\n*   LTA to refund S$1.4m in extra road tax collected from commercial vehicles after 'discrepancy' discovered\n*   Singapore Sports Hub renamed The Kallang to honour country's sporting heritage\n*   New ferry service connects Singapore and Batam’s Gold Coast terminal\n*   Singapore must develop robotics, AI talent pipeline to stay competitive: National Robotics Programme\n*   Buskers can now perform together under NAC pilot; new spots added across Singapor